# Building Multi-Agent Systems with CrewAI

A minimal, working CrewAI setup that runs on **Groq** (free, fast inference).

This notebook builds a small two-agent *crew* — a **Researcher** that gathers key
points on a topic, and a **Writer** that turns those points into a short brief.

**How to run:** use *Run All* (top toolbar) so every cell executes top-to-bottom
in order. Running a single cell in isolation will fail because later cells depend
on variables defined earlier.

## 1. Install dependencies

`crewai` pulls in `litellm` (the layer that talks to Groq) and everything else.
`python-dotenv` loads your API key from a local `.env` file.

In [ ]:
%pip install -q crewai python-dotenv

## 2. Load the API key and configure the LLM

Create a file named `.env` next to this notebook containing:

```
GROQ_API_KEY=your_real_key_here
```

Get a free key at https://console.groq.com/keys. The `.env` file is git-ignored,
so your key never gets committed.

In [ ]:
from dotenv import load_dotenv
import os
from crewai import LLM

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Create a .env file next to this notebook "
        "with a line like:  GROQ_API_KEY=your_key_here"
    )

# Groq models are addressed as "groq/<model-name>" via LiteLLM.
llm = LLM(model="groq/llama-3.1-8b-instant", api_key=api_key)

print("LLM configured:", llm.model)

## 3. Compatibility shim for Groq

CrewAI 1.15.x tags every message with a `cache_breakpoint` field — a prompt-caching
hint that only Anthropic's API understands. When you route through Groq, that extra
field is sent verbatim and Groq's strict schema rejects it:

> `property 'cache_breakpoint' is unsupported`

The line below neutralizes the tagging so the same code works on any provider.
Remove it once you switch to Anthropic or once CrewAI fixes this upstream.

In [ ]:
import crewai.llms.cache as _crewai_cache

# Make the cache-breakpoint tag a no-op so non-Anthropic providers accept the request.
_crewai_cache.mark_cache_breakpoint = lambda message: message

print("Groq compatibility shim applied.")

## 4. Define the agents

Each agent has a **role**, a **goal**, and a **backstory** that shapes how it behaves.
Both agents share the same `llm` we configured above.

In [ ]:
from crewai import Agent

researcher = Agent(
    role="Research Analyst",
    goal="Find and summarize the most important, current facts about {topic}",
    backstory=(
        "You are a meticulous analyst who distills noisy information into a short "
        "list of accurate, high-signal bullet points."
    ),
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Technical Writer",
    goal="Turn research notes about {topic} into a clear, engaging short brief",
    backstory=(
        "You are a writer who turns dense notes into crisp prose that a busy "
        "reader can absorb in under a minute."
    ),
    llm=llm,
    verbose=True,
)

## 5. Define the tasks

Tasks describe *what* to do and the *expected output*. The `{topic}` placeholder is
filled in at run time from the `inputs` dict. The writer's task lists the research
task in `context`, so the researcher's output is passed to the writer automatically.

In [ ]:
from crewai import Task

research_task = Task(
    description="Research {topic} and list the key points a newcomer should know.",
    expected_output="4-6 concise bullet points, each one fact or insight.",
    agent=researcher,
)

writing_task = Task(
    description="Using the research notes, write a short brief on {topic}.",
    expected_output="A 2-3 paragraph brief in plain, engaging language.",
    agent=writer,
    context=[research_task],
)

## 6. Assemble and run the crew

`Process.sequential` runs tasks in order: research first, then writing.

Jupyter already runs an asyncio event loop, so we use `await crew.kickoff_async(...)`
instead of the synchronous `crew.kickoff(...)` (which raises inside a running loop).

In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

result = await crew.kickoff_async(inputs={"topic": "multi-agent AI systems"})

print()
print("=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

## 7. Inspect the run

`result` is a `CrewOutput` object. You can pull out each task's individual output
and the token usage for the whole run.

In [ ]:
print("Token usage:", result.token_usage)
print()
for i, task_output in enumerate(result.tasks_output, start=1):
    print(f"--- Task {i}: {task_output.agent} ---")
    print(task_output.raw)
    print()